# Risk indicators — catalog & behaviour (offline)

Inspect the bundled dataset catalogue and the backend's design rules
**without touching the network**: which datasets ship, their provider and
output kind, the per-instance `OUTPUT_KIND` (tabular vs vector), the
did-you-mean hint on a bad id, and why `aggregate=` is rejected. This is
the no-network companion to the live
[quickstart](01_risk_indicators_quickstart.ipynb).

In [1]:
import tempfile

import pandas as pd
from earthlens.earthlens import EarthLens

## The shipped datasets

`EarthLens.list_datasets("risk-indicators")` lists every dataset id the
backend accepts in `variables=` — no construction, no network.

In [2]:
EarthLens.list_datasets("risk-indicators")

2026-06-27 04:27:46 | INFO | pyramids.base.config | Logging is configured.


['gfw:admin_boundary',
 'gfw:tree_cover_loss',
 'gfw:tree_cover_loss_summary',
 'inform:climate_risk',
 'inform:coping_capacity',
 'inform:hazard_exposure',
 'inform:risk',
 'inform:vulnerability',
 'thinkhazard:all',
 'thinkhazard:cyclone',
 'thinkhazard:earthquake',
 'thinkhazard:extreme_heat',
 'thinkhazard:flood_coastal',
 'thinkhazard:flood_river',
 'thinkhazard:flood_urban',
 'thinkhazard:landslide',
 'thinkhazard:tsunami',
 'thinkhazard:volcano',
 'thinkhazard:water_scarcity',
 'thinkhazard:wildfire']

## Catalogue rows — provider and output kind

`EarthLens.catalog("risk-indicators")` returns the bundled `Catalog`.
Each row is a frozen pydantic `Dataset` carrying its `provider`,
`output_kind`, and human label. The ids group cleanly by source —
`thinkhazard:*`, `inform:*`, `gfw:*`.

In [3]:
catalog = EarthLens.catalog("risk-indicators")
pd.DataFrame(
    [
        {
            "id": i,
            "provider": catalog.get(i).provider,
            "output_kind": catalog.get(i).output_kind,
            "long_name": catalog.get(i).long_name,
        }
        for i in catalog.available()
    ]
).set_index("id")

,provider,output_kind,long_name
id,,,
gfw:admin_boundary,gfw,vector,GADM admin boundary geometry the GFW indicator...
gfw:tree_cover_loss,gfw,tabular,Annual tree-cover loss (ha) by country (UMD/Ha...
gfw:tree_cover_loss_summary,gfw,tabular,Total tree-cover loss (ha) by country (UMD/Han...
inform:climate_risk,inform,tabular,INFORM Climate Change Risk (SSP5 2050)
inform:coping_capacity,inform,tabular,INFORM Lack of Coping Capacity dimension
inform:hazard_exposure,inform,tabular,INFORM Hazard & Exposure dimension
inform:risk,inform,tabular,INFORM Risk composite index
inform:vulnerability,inform,tabular,INFORM Vulnerability dimension
thinkhazard:all,thinkhazard,tabular,All ThinkHazard! hazard levels for a division


## Per-instance `OUTPUT_KIND`

The backend's return shape is **decided per dataset**, not fixed for the
whole backend. A `thinkhazard:*` / `inform:*` / `gfw:tree_cover_loss` row
is `tabular` → it returns a `pandas.DataFrame`; `gfw:admin_boundary` is
`vector` → it returns a pyramids `FeatureCollection`. The facade reads the
resolved dataset's `output_kind` onto the instance `OUTPUT_KIND` to know
the return shape (and to gate `aggregate=`).

In [4]:
pd.DataFrame(
    [
        {"id": i, "output_kind": catalog.get(i).output_kind}
        for i in [
            "thinkhazard:flood_river",
            "inform:risk",
            "gfw:tree_cover_loss",
            "gfw:admin_boundary",
        ]
    ]
).set_index("id")

,output_kind
id,
thinkhazard:flood_river,tabular
inform:risk,tabular
gfw:tree_cover_loss,tabular
gfw:admin_boundary,vector


## Resolving ids — with a did-you-mean hint

An unknown id raises a `ValueError` that suggests the closest match,
rather than failing silently. (The cell below is expected to raise — it
carries the `raises-exception` tag so notebook execution still passes.)

In [5]:
EarthLens.catalog("risk-indicators").get("inform:rsk")

ValueError: 'inform:rsk' is not in the risk-indicators catalog. Known datasets: ['gfw:admin_boundary', 'gfw:tree_cover_loss', 'gfw:tree_cover_loss_summary', 'inform:climate_risk', 'inform:coping_capacity', 'inform:hazard_exposure', 'inform:risk', 'inform:vulnerability', 'thinkhazard:all', 'thinkhazard:cyclone', 'thinkhazard:earthquake', 'thinkhazard:extreme_heat', 'thinkhazard:flood_coastal', 'thinkhazard:flood_river', 'thinkhazard:flood_urban', 'thinkhazard:landslide', 'thinkhazard:tsunami', 'thinkhazard:volcano', 'thinkhazard:water_scarcity', 'thinkhazard:wildfire']. Did you mean 'inform:risk'?

## `aggregate=` is rejected

These are pre-computed country-indexed indices, not gridded rasters, so
there is no meaningful gridded reduction: passing a non-`None`
`aggregate=` raises `NotImplementedError`. The guard fires in the facade
**before** any network call (the `inform:risk` backend constructs without
touching the network). Again shown via a `raises-exception`-tagged cell.

In [6]:
EarthLens(
    data_source="inform",
    variables=["inform:risk"],
    country="KEN",
    path=tempfile.mkdtemp(),
).download(aggregate=object())

NotImplementedError: aggregate= is not supported for RiskIndicators backends (OUTPUT_KIND='tabular'). The aggregator only handles gridded raster outputs; vector / tabular backends emit GeoDataFrames or DataFrames that do not have a meaningful gridded reduction.

## Takeaway

- `EarthLens.list_datasets("risk-indicators")` and
  `EarthLens.catalog(...)` enumerate the shipped datasets and their
  metadata with no network.
- `OUTPUT_KIND` is **per dataset**: tabular ids return a `DataFrame`,
  `gfw:admin_boundary` returns a `FeatureCollection`.
- A bad id raises with a did-you-mean hint; `aggregate=` is rejected by
  design. See the [quickstart](01_risk_indicators_quickstart.ipynb) and
  [GFW notebook](03_global_forest_watch.ipynb) for the live calls.